In [12]:
import duckdb
import pandas as pd

con = duckdb.connect("../notebooks/lakehouse.duckdb")
con.execute("SHOW TABLES").df()

,name
0,raw_customers
1,raw_orders
2,raw_products
3,stage_customers
4,stage_orders
5,stage_products


In [13]:
# DIM_CLIENTE: atributos descriptivos del cliente, sin cálculos
con.execute("""
    CREATE OR REPLACE TABLE dim_cliente AS
    SELECT customer_id, full_name, email, city, age, registration_date
    FROM stage_customers
""")

# DIM_PRODUCTO: atributos descriptivos del producto
con.execute("""
    CREATE OR REPLACE TABLE dim_producto AS
    SELECT product_id, product_name, category, price
    FROM stage_products
""")

# FACT_CLIENTE: métricas calculadas por cliente, agregando sus pedidos.
# LEFT JOIN para que TODOS los clientes aparezcan, incluso los que
# nunca compraron (quedan en 0, no desaparecen de la tabla)
con.execute("""
    CREATE OR REPLACE TABLE fact_cliente AS
    SELECT
        c.customer_id,
        COALESCE(COUNT(o.order_id), 0) AS total_pedidos,
        COALESCE(SUM(o.total_amount_usd), 0) AS monto_total_gastado,
        COALESCE(AVG(o.total_amount_usd), 0) AS ticket_promedio,
        MAX(o.order_date) AS fecha_ultimo_pedido
    FROM dim_cliente c
    LEFT JOIN stage_orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_id
""")

# FACT_PRODUCTO: métricas calculadas por producto
con.execute("""
    CREATE OR REPLACE TABLE fact_producto AS
    SELECT
        p.product_id,
        COALESCE(SUM(o.quantity), 0) AS unidades_vendidas,
        COALESCE(SUM(o.total_amount_usd), 0) AS ingresos_totales,
        COALESCE(COUNT(o.order_id), 0) AS num_pedidos
    FROM dim_producto p
    LEFT JOIN stage_orders o ON p.product_id = o.product_id
    GROUP BY p.product_id
""")

con.execute("SHOW TABLES").df()

,name
0,dim_cliente
1,dim_producto
2,fact_cliente
3,fact_producto
4,raw_customers
5,raw_orders
6,raw_products
7,stage_customers
8,stage_orders
9,stage_products


In [14]:
con.execute("SELECT * FROM fact_cliente LIMIT 5").df()

,customer_id,total_pedidos,monto_total_gastado,ticket_promedio,fecha_ultimo_pedido
0,1,0,0.0,0.00,NaT
1,2,0,0.0,0.00,NaT
2,3,0,0.0,0.00,NaT
3,4,4,668.4,167.10,2025-02-19
4,5,2,79.5,39.75,2025-05-01


In [15]:
con.execute("SELECT * FROM fact_producto LIMIT 5").df()

,product_id,unidades_vendidas,ingresos_totales,num_pedidos
0,1,22.0,4148.5,6
1,4,7.0,99.4,3
2,5,4.0,49.9,2
3,6,8.0,519.9,2
4,7,4.0,118.9,2


In [16]:
con.close()

## Esquema de la capa ANALYTICS

### FACT_CLIENTE
| Columna | Tipo | Descripción |
|---|---|---|
| customer_id | INTEGER | Identificador único del cliente (FK a DIM_CLIENTE) |
| total_pedidos | INTEGER | Cantidad de pedidos realizados por el cliente |
| monto_total_gastado | DOUBLE | Suma de todos los montos gastados por el cliente |
| ticket_promedio | DOUBLE | Promedio del monto gastado por pedido |
| fecha_ultimo_pedido | DATE | Fecha del pedido más reciente del cliente |

### FACT_PRODUCTO
| Columna | Tipo | Descripción |
|---|---|---|
| product_id | INTEGER | Identificador único del producto (FK a DIM_PRODUCTO) |
| unidades_vendidas | DOUBLE | Total de unidades vendidas del producto |
| ingresos_totales | DOUBLE | Suma de ingresos generados por el producto |
| num_pedidos | INTEGER | Cantidad de pedidos que incluyeron este producto |

### DIM_CLIENTE
| Columna | Tipo | Descripción |
|---|---|---|
| customer_id | INTEGER | Identificador único del cliente |
| full_name | VARCHAR | Nombre completo del cliente |
| email | VARCHAR | Correo electrónico del cliente |
| city | VARCHAR | Ciudad de residencia |
| age | INTEGER | Edad del cliente |
| registration_date | DATE | Fecha de registro del cliente |

### DIM_PRODUCTO
| Columna | Tipo | Descripción |
|---|---|---|
| product_id | INTEGER | Identificador único del producto |
| product_name | VARCHAR | Nombre del producto |
| category | VARCHAR | Categoría del producto |
| price | DOUBLE | Precio unitario del producto |